In [0]:
%run ../Includes/common_functions

In [0]:
%run ../Includes/config

In [0]:
v_data_source = dbutils.widgets.get("p_data_source")
v_file_date = dbutils.widgets.get("p_file_date")
print(v_data_source)
print(v_file_date)
print(raw_folder_path)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
results_schema = StructType(fields=[StructField("resultId", StringType(), False),
                                    StructField("raceId", StringType(), True),
                                    StructField("driverId", StringType(), True),
                                    StructField("constructorId", StringType(), True),
                                    StructField("number", StringType(), True),
                                    StructField("grid", StringType(), True),
                                    StructField("position", StringType(), True),
                                    StructField("positionText", StringType(), True),
                                    StructField("positionOrder", StringType(), True),
                                    StructField("points", StringType(), True),
                                    StructField("laps", StringType(), True),
                                    StructField("time", StringType(), True),
                                    StructField("milliseconds", StringType(), True),
                                    StructField("fastestLap", StringType(), True),
                                    StructField("rank", StringType(), True),
                                    StructField("fastestLapTime", StringType(), True),
                                    StructField("fastestLapSpeed", StringType(), True),
                                    StructField("statusId", StringType(), True)])

# COMMAND ----------

results_df = (spark.read 
.schema(results_schema)
.format("json")
.load(f"{raw_folder_path}/{v_file_date}/results.json"))

In [0]:
from pyspark.sql.functions import current_timestamp,lit
results_with_columns_df = (results_df.withColumnRenamed("resultId", "result_id") 
                                    .withColumnRenamed("raceId", "race_id") 
                                    .withColumnRenamed("driverId", "driver_id")
                                    .withColumnRenamed("constructorId", "constructor_id") 
                                    .withColumnRenamed("positionText", "position_text") 
                                    .withColumnRenamed("positionOrder", "position_order") 
                                    .withColumnRenamed("fastestLap", "fastest_lap") 
                                    .withColumnRenamed("fastestLapTime", "fastest_lap_time") 
                                    .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed") 
                                    .withColumn("ingestion_date", current_timestamp())
                                    .withColumn("data_source", lit(v_data_source)) 
                                    .withColumn("file_date", lit(v_file_date))
                                    ) 
                            

In [0]:
from pyspark.sql.functions import col

results_final_df = results_with_columns_df.drop(col("statusId"))

(results_final_df.write.mode("overwrite")
.format("delta")
.option("mergeSchema","true")
.saveAsTable("f1.Bronze.results"))

dbutils.notebook.exit("Success")